# Topography Notes
These are notes on how to create a topographic / hypsometric map of Germany. This was a hard piece of work, and I had a lot of trouble to find a way to do so; especially finding out how the result could be exported as SVG with the isobands / elevation levels as (Multi)Polygons. Since I'm not sure whether I could use some of the approaches again, this is my notebook to save them.

## Data: Where to find it

The data format of topographic data is generally a raster, where each pixel describes the elevation at its area. The file format is mainly *.tif*. There are several sources and I downloaded different files:

### SRTM
 I don't remember where and how I downloaded it (maybe [here](https://portal.opentopography.org/raster?opentopoID=OTSRTM.082015.4326.1)?)

### Copernicus
 I downloaded the data using the [Copernicus Browser](https://browser.dataspace.copernicus.eu/) (login required)
- On the left-hand side, under "DATA COLLECTIONS", choose "Copernicus DEM".
- Then you can choose between "Copernicus 30" and "Copernicus 90"; I use 90 (lower resolution), but you can also use 30 (higher resolution).
- For "LAYERS" I used "Topographic".
- On the right-hand side, use the Polygon-tool (Icon with tooltip "Create an area of interest") and select a rectangle on the map. This was a little annoying, since no large areas can be chosen; but I fiddled around to get a rectangle of Germany.
- Zoom in, and then on the right-hand side, use the Download-tool (Icon with tooltip "Download image").
    - On the top, go to "Analytical".
    - Image format: TIFF (bit size does not really matter).
    - Image resolution: choose what you want.
    - Coordinate system: Whatever you want; "EPSG: 4326" might be best.
    - Remove ticks from "Visualised"; set tick at "Raw": "DEM"
    - Click download on top right.

## Data: How to open it
The best tool for opening data ist *rasterio*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from shapely.geometry import MultiPolygon, Polygon

from geofeatureviz import map_style
from geofeatureviz.io import path_settings

In [ ]:
# file_path = path_settings.data_dir / "srtm_germany_dtm.tif"
file_path = path_settings.data_dir / "COPERNICUS_90_DEM.tiff"
with rasterio.open(file_path) as raster_src:
    # get the elevation raster as 2x2 array
    elevation = raster_src.read(1)
    # get the reference system / projection
    crs = raster_src.crs
    # get a matrix to transform from pixel space to geo space
    transform = raster_src.transform

fig, axes = plt.subplots(1, 2, constrained_layout=True, sharex=True, sharey=True)
cmap = map_style.get_above_sea_level_cmap()

# show elevation map
img = axes[0].imshow(elevation, cmap=cmap)
# show contour lines with matplotlib (has to be turned upside down!)
axes[1].contour(elevation, cmap=cmap)
axes[1].set_aspect("equal")

cbar = fig.colorbar(img, ax=axes[1], location="right")
cbar.set_label("Elevation (m)")

plt.show()

Note that in the plots, the y-axis is upside-down, which makes the image look correctly. In a lot of cases, you have to use `elevations[::-1]` to not have the map upside down.

### Crop to Germany

In [ ]:
from rasterio.mask import mask as rasterio_mask

from geofeatureviz_scripts import loader

In [ ]:
# get Germany's geometry
germany = loader.load_and_prep_data("country", "ne", 10)
germany = germany[germany["admin"] == "Germany"]
germany_geom = [germany.geometry.values[0]]

# crop to Germany
file_path = path_settings.data_dir / "COPERNICUS_90_DEM.tiff"
with rasterio.open(file_path) as raster_src:
    elev_masked, elev_transform = rasterio_mask(
        dataset=raster_src, shapes=germany_geom, crop=True
    )
elev_masked = elev_masked[0]

# Step 4: Plot topographic map
plt.imshow(elev_masked, cmap=cmap)
plt.colorbar(label="Elevation (m)")
plt.show()

## GDAL
GDAL (`sudo apt install gdal-bin`; I uninstalled it in the meantime) is a tool for geospatial data and can compute vectors from raster data. I had two approaches here:

### Contour Vector
I used it to create a geopackage with vector data from the raster. This did not really make sense, since I tried to create Polygons from contours without further thought. I think I used the following command:
```bash
 gdal_contour -i 50 COPERNICUS_90_DEM.tiff COPERNICUS_90_DEM.gpkg -a elevation
```

In [ ]:
import geopandas as gpd

In [ ]:
gdf = gpd.read_file(path_settings.data_dir / "unused" / "COPERNICUS_90_DEM.gpkg")

gdf = gdf.dissolve(by="elevation").reset_index()

fig, ax = plt.subplots()
ax.set_axis_off()
ax.set_aspect("equal")
gdf.plot(ax=ax)

plt.show()
# plt.savefig("topographic_map.svg", format="svg", bbox_inches="tight")

### More dedicated Polygons from Contours
First, I smoothened the raster and then tried to create a polygon for several elevation levels.
```bash
gdalwarp -r cubic COPERNICUS_90_DEM.tiff dem_smooth.tif

gdal_calc.py -A dem_smooth.tif --outfile=gt0.tif --calc="A>0"
gdal_calc.py -A dem_smooth.tif --outfile=gt100.tif --calc="A>100"
gdal_calc.py -A dem_smooth.tif --outfile=gt200.tif --calc="A>200"
gdal_calc.py -A dem_smooth.tif --outfile=gt500.tif --calc="A>500"
gdal_calc.py -A dem_smooth.tif --outfile=gt1000.tif --calc="A>1000"

gdal_polygonize.py gt0.tif   g0.geojson
gdal_polygonize.py gt100.tif g100.geojson
gdal_polygonize.py gt200.tif g200.geojson
gdal_polygonize.py gt500.tif g500.geojson
gdal_polygonize.py gt1000.tif g1000.geojson
```

And then loaded them as GeoDataFrames and plotted them together. Still, the problem remains, that contours are not the perfect way to create polygons.

In [ ]:
elevation_levels = [0, 100, 200, 500, 1000]
gdfs = [gpd.read_file(f"g{level}.tif") for level in elevation_levels]

fig, ax = plt.subplots()

colors = ["green", "yellow", "orange", "red", "brown"]
for gdf, color in zip(gdfs, colors):
    gdf.plot(ax=ax, color=color)
plt.show()

## Hillshades with Matplotlib

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from matplotlib.colors import LightSource

path = path_settings.data_dir / "COPERNICUS_90_DEM.tiff"
with rasterio.open(path) as src:
    elevation = src.read(1)
    elevation = np.where(elevation == src.nodata, np.nan, elevation)

# create hillshade
ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(elevation, vert_exag=1)

fig, ax = plt.subplots()
img = ax.imshow(elevation, cmap=cmap)

# hillshade overlay (gives relief effect)
ax.imshow(hillshade, cmap="gray", alpha=0.4)

# plt.savefig("topographic_map.svg", format="svg", bbox_inches="tight")
plt.show()

## Create Polygons with Rasterio
Rasterio has a built-in function to create shapes from the raster. The problem is, that this function produces grid-artifacts (stair-case like shapes). A solution would be to smoothen the polygons afterward; this is not the best solution though, since a lot of coordinate information is lost during polygon creation. Just to remember the name, one algorithm to smoothen the polygon would be the Chaikin-algorithm, but this makes the polygon smaller over its internal iterations, that's why I did not dig deeper here.

In [ ]:
from rasterio.features import shapes as rasterio_shapes
from shapely.geometry import shape as shapely_shape

In [ ]:
path = path_settings.data_dir / "COPERNICUS_90_DEM.tiff"
with rasterio.open(path) as src:
    elevation = src.read(1).astype("float64")
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

# mask nodata
mask = (elevation != nodata) & ~np.isnan(elevation)

# convert DEM to class index
bins = [0, 100, 200, 500, 1000, 2000, 3000]
zones = np.digitize(elevation, bins)

# polygonize
results = rasterio_shapes(zones.astype("int16"), mask=mask, transform=transform)

# create dataframe
geoms = []
vals = []
for geom, value in results:
    geoms.append(shapely_shape(geom))
    vals.append(value)
gdf = gpd.GeoDataFrame({"zone": vals}, geometry=geoms, crs=crs)

# plot
fig, ax = plt.subplots()
ax.set_aspect("equal")

gdf.plot(column="zone", cmap=cmap, linewidth=0, edgecolor="none", ax=ax)

# plt.savefig("elevation_map.svg", format="svg", bbox_inches="tight")
plt.show()

One approach to smoothen geoms is to use a buffer, but this can change its topography:

In [ ]:
def smoothen_geom(geom: Polygon | MultiPolygon, r: float = 1):
    """Make a geometry smoother by buffering it outwards and then inwards."""
    if isinstance(geom, Polygon):
        return geom.buffer(r).buffer(-r)
    if isinstance(geom, MultiPolygon):
        geoms = [g.buffer(r).buffer(-r) for g in geom.geoms]
        return MultiPolygon(geoms)

## Create Polygons from Contours
My first approach on this was using Matplotlibs contours, and then create polygons from it. However, contours do not really create polygons; they are only lines showing a contour. More processing would be necessary to create polygons from contours.

In [ ]:
path = path_settings.data_dir / "COPERNICUS_90_DEM.tiff"
with rasterio.open(path) as src:
    elevation = src.read(1).astype("float32")
    elevation = elevation[::-1]
    elevation[elevation == src.nodata] = np.nan

# fill NaNs for interpolation (important for contouring)
elevation = np.nan_to_num(elevation, nan=np.nanmean(elevation))

# create contours
fig, ax = plt.subplots()
levels = [0, 100, 200, 500, 1000, 2000, 3000]
cs = ax.contourf(elevation, levels=levels, cmap="terrain")

# extract polygons from contourf paths
polygons = []
values = []
for i, segs in enumerate(cs.allsegs):
    for seg in segs:
        if len(seg) < 3:
            continue

        poly = Polygon(seg)
        if poly.is_valid:
            polygons.append(poly)
            values.append(i)
plt.close()

# build GeoDataFrame
gdf = gpd.GeoDataFrame({"zone": values}, geometry=polygons, crs="EPSG:4326")

# optional dissolve (keeps zones clean)
gdf = gdf.dissolve(by="zone").reset_index()

fig, ax = plt.subplots()
ax.set_aspect("equal")
gdf.plot(column="zone", cmap="terrain", linewidth=0, edgecolor="none", ax=ax)

# plt.savefig("hypsometric_map.svg", format="svg", bbox_inches="tight")
plt.show()

## Find contours with skimage
This was another approach to get the contours, but had the same problem as previous approaches; for example holes are a problem.

In [ ]:
from affine import Affine
from rasterio import transform as rasterio_transform
from scipy.ndimage import gaussian_filter
from shapely.ops import unary_union
from skimage import measure

In [ ]:
def raster_to_polygons(raster, threshold, transform):
    """Create polygons from a raster by thresholding and contouring."""
    mask = (raster > threshold).astype(float)
    smoothed = gaussian_filter(mask, sigma=0.5)
    contours = measure.find_contours(smoothed, level=0.5)

    polygons = []
    for contour in contours:
        coords = []
        for row, col in contour:
            x, y = rasterio_transform.xy(transform, row, col)
            coords.append((x, y))

        if len(coords) < 3:
            continue

        poly = Polygon(coords)
        if poly.is_valid:
            polygons.append(poly)
    return unary_union(polygons)


grid = np.array(
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        # main shape: a "blocky circle-like blob"
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    dtype=np.uint8,
)

transform = Affine.scale(1, -1)
g = raster_to_polygons(grid, 0.5, transform)
g

## GRASS
GRASS is a CLI-tool with raster and vector functionality for GIS. I tried to use it to 

The CLI can be started by simply running `grass`. All variables are permanently stored inside GRASS's environment. I tried to recreate the stuff I did, it was something along the lines of:
```bash
r.import input=data/COPERNICUS_90_DEM.tiff output=dem
r.resamp.interp input=dem output=dem_fine method=bicubic
r.recode input=dem_fine output=zones rules=rules.txt
r.to.vect input=zones output=isobands type=area
v.db.addcolumn map=isobands columns="min_elev double precision, max_elev double precision"

v.generalize input=isobands output=isobands_smooth method=sliding_averaging threshold=8
v.clean input=isobands_smooth output=isobands_clean tool=rmarea,prune --overwrite
v.out.ogr input=isobands_clean output=isobands.gpkg format=GPKG --overwrite
```

where rules.txt looked like:
```text
0:100:1
100:200:2
200:300:3
300:400:4
400:500:5
500:600:6
600:700:7
700:800:8
800:900:9
900:1000:10
1000:1100:11
1100:1200:12
1200:1300:13
1300:1400:14
1400:1500:15
1500:1600:16
1600:1700:17
1700:1800:18
1800:1900:19
1900:2000:20
2000:2100:21
2100:2200:22
2200:2300:23
2300:2400:24
2400:2500:25
2500:2600:26
2600:2700:27
2700:2800:28
2800:2900:29
2900:3000:30
```

But in the end, I did not really use this geopackage (which could be imported with geopandas).

## Final Approach: Contourpy
My final approach is to use contourpy, also a library to create contours. However, it also provides information on areas to fill, so you can create polygons with holes from it. This is the code generated by Claude; I removed some stuff and rewrote it for my purpose. Especially all the part at the end (assigning holes to their outers with sorting and stuff) is not necessary from what I've seen.

In [ ]:
import contourpy
import numpy as np
from shapely.geometry import Point, Polygon
from shapely.ops import transform as shapely_transform

In [ ]:
_MOVETO, _CLOSEPOLY = 1, 79


def raster_to_polygons(raster, threshold, band=1):
    """Convert a single-band elevation raster into a shapely (Multi)Polygon.

    The Polygon covers all area with elevation >= threshold, using marching squares
    (contourpy) so edges are smooth/sub-pixel rather than following raster
    cell boundaries.

    raster: an open rasterio DatasetReader
    threshold: elevation value; resulting polygon covers elevation >= threshold
    band: band index to read (default 1)

    Returns a shapely Polygon, MultiPolygon, or None if nothing is >= threshold.
    """
    elevation = raster.read(band).astype("float64")
    if raster.nodata is not None:
        elevation = np.where(elevation == raster.nodata, np.nan, elevation)

    cg = contourpy.contour_generator(z=elevation, fill_type="OuterCode")
    points_list, codes_list = cg.filled(threshold, np.inf)

    rings = []
    for points, codes in zip(points_list, codes_list):
        if points is None:
            continue
        start = None
        for i, c in enumerate(codes):
            if c == _MOVETO:
                start = i
            elif c == _CLOSEPOLY:
                rings.append(points[start : i + 1])

    if not rings:
        return None

    def signed_area(ring):
        x, y = ring[:, 0], ring[:, 1]
        return 0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])

    outers, holes = [], []
    for r in rings:
        (outers if signed_area(r) > 0 else holes).append(r)

    outer_polys = [Polygon(o) for o in outers]
    order = np.argsort([p.area for p in outer_polys])[::-1]
    outer_polys = [outer_polys[i] for i in order]

    hole_bins = [[] for _ in outer_polys]
    for h in holes:
        pt = Point(h.mean(axis=0))
        candidates = [i for i, op in enumerate(outer_polys) if op.contains(pt)]
        if candidates:
            best = min(candidates, key=lambda i: outer_polys[i].area)
            hole_bins[best].append(h)

    polys = []
    for op, hlist in zip(outer_polys, hole_bins):
        poly = Polygon(op.exterior.coords, holes=hlist)
        if not poly.is_valid:
            poly = poly.buffer(0)
        if not poly.is_empty:
            polys.append(poly)

    if not polys:
        return None

    result = unary_union(polys)

    # pixel index (col, row) -> map coords. contourpy's default coordinates
    # equal array indices directly (index N = pixel-N center), while
    # rasterio's affine transform maps index (col,row) -> pixel's
    # upper-left corner, so add 0.5 to land on the pixel center.
    transform = raster.transform
    return shapely_transform(lambda x, y: transform * (x + 0.5, y + 0.5), result)


data_file = path_settings.data_dir / "COPERNICUS_90_DEM.tiff"
thresholds = np.arange(0, 3200, 200)

geom_dicts = []
with rasterio.open(data_file) as src:
    for t in thresholds:
        geom = raster_to_polygons(src, t)
        geom_dicts.append({"elevation": t, "geometry": geom})
gdf = gpd.GeoDataFrame(geom_dicts)

# assign colors
topo_colors = [c for level, c in map_style.get_topo_colors().items() if level > 0][::-1]
idx = np.round(np.linspace(0, len(topo_colors) - 1, len(thresholds))).astype(int)
colors = dict(zip(thresholds, np.array(topo_colors)[idx]))
gdf["color"] = gdf["elevation"].map(colors)

gdf.plot(color=gdf["color"])
# plt.savefig("hypsometric_map.svg")
plt.show()